In [ ]:
# One-cell external asset-forensics runner (all outputs remain private in Google Drive)
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, pathlib, ipywidgets as widgets
from IPython.display import display, clear_output

REPO = "https://github.com/ZzwahwahzZ/Clash-of-Legends.git"
BRANCH = "tools/colab-drive-asset-transfer"
WORK = pathlib.Path("/content/clash-of-legends-forensics")
OUT_ROOT = pathlib.Path("/content/drive/MyDrive/ClashOfLegends_Forensics")

print("Installing independent open tools: 7-Zip, libarchive, binwalk, ffmpeg, Blender...")
subprocess.run(["bash","-lc","apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full libarchive-tools binwalk ffmpeg blender"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","ipyfilechooser","kaitaistruct"],check=True)

if WORK.exists():
    subprocess.run(["git","-C",str(WORK),"fetch","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(WORK),"checkout",BRANCH],check=True)
    subprocess.run(["git","-C",str(WORK),"pull","--ff-only","origin",BRANCH],check=True)
else:
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO,str(WORK)],check=True)

from ipyfilechooser import FileChooser
chooser = FileChooser("/content/drive/MyDrive")
chooser.title = "<b>اختاري أرشيف المصدر الأصلي يدويًا من Google Drive</b>"
chooser.filter_pattern = ["*.zip","*.7z","*.rar","*.tar","*.tgz","*.apk","*.obb"]
chooser.show_only_dirs = False
run_button = widgets.Button(description="ابدئي الفحص والاستخراج",button_style="success",icon="search")
status = widgets.Output()

def run_pipeline(_):
    with status:
        clear_output()
        selected = chooser.selected
        if not selected or not pathlib.Path(selected).is_file():
            print("اختاري ملف أرشيف صحيح الأول.")
            return
        source = pathlib.Path(selected)
        destination = OUT_ROOT / source.stem
        destination.mkdir(parents=True,exist_ok=True)
        scanner = WORK / "tools/asset_forensics/run_forensics.py"
        print("المصدر:", source)
        print("مكان النتائج الخاص:", destination)
        cmd = [sys.executable,str(scanner),str(source),"--out",str(destination)]
        result = subprocess.run(cmd,text=True)
        print("\nExit code:",result.returncode)
        report = destination / "forensic-report.json"
        candidates = destination / "resource-candidates.csv"
        print("التقرير:",report)
        print("فهرس الموارد:",candidates)
        print("الملفات المستخرجة:",destination / "extracted")
        print("\nالمرحلة التالية: فحص ملفات C3 / skeleton / motion / FUI، ثم تحويل الموديلات القياسية إلى GLB وفحص الصوتيات.")
        if report.exists():
            import json
            data=json.loads(report.read_text(encoding="utf-8"))
            print("\nملخص:")
            print("عدد الملفات:",data.get("file_count"))
            print("التصنيفات:",data.get("category_counts"))
            print("الأرشيفات:",data.get("archives"))
            print("أخطاء:",data.get("errors"))

run_button.on_click(run_pipeline)
display(chooser,run_button,status)
